In [ ]:
!echo $CONDA_DEFAULT_ENV

In [ ]:
import os
import sys
import glob
import re
import random
import math
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import xesmf as xe
import cf_xarray
import cordex as cx
import pyproj

import geopandas as gpd
from shapely.geometry import Point
from shapely import vectorized, contains_xy

import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm

from evaltools.source import xarray_open_kwargs

from urclimask.GHCNd_stations import get_valid_timeseries, inside_city
from urclimask.UHI_analysis import UrbanIsland
from urclimask.urban_areas import UrbanVicinity, plot_urban_polygon
from urclimask.utils import load_ucdb_city, traverseDir

from tools import (
    check_equal_period,
    fix_360_longitudes,
    open_datasets,
    standardize_unit,
)

In [ ]:
# =============================================================================
# GEOMETRY AND DISTANCE UTILITIES
# =============================================================================

def haversine(lon1: float, lat1: float, lon2: float, lat2: float) -> float:
    """
    Calculate the great-circle distance (in kilometers) between two points 
    on the Earth using the Haversine formula.
    """
    R = 6371  # Earth radius in kilometers
    
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    
    return 2 * R * asin(sqrt(a))

def create_urban_mask(ds: xr.Dataset, polygon, var_name: str = "urmask") -> xr.DataArray:
    """
    Create an urban mask (1 for cells inside the polygon, 0 otherwise) 
    for the grid defined by the input Dataset.
    
    Args:
        ds: The Xarray Dataset defining the grid (must have 'lat' and 'lon' coordinates).
        polygon: A shapely Polygon object defining the urban area.
        var_name: The name for the output DataArray.

    Returns:
        An Xarray DataArray representing the mask.
    """
    lats = ds.lat.values
    lons = ds.lon.values

    # Initialize mask array with zeros
    mask = np.zeros((len(lats), len(lons)), dtype=np.int16)

    # Simple nested loop iteration for masking (can be slow for large grids)
    for i, la in enumerate(lats):
        for j, lo in enumerate(lons):
            p = Point(lo, la)
            if polygon.contains(p):
                mask[i, j] = 1

    # Create DataArray with the mask and coordinates
    urmask = xr.DataArray(mask, coords=[("lat", lats), ("lon", lons)], name=var_name)
    return urmask

# =============================================================================
# TEXT AND NAME NORMALIZATION UTILITIES
# =============================================================================

def clean_text(s: str) -> str:
    """Remove invisible characters (including BOM) from a string."""
    if s is None or pd.isna(s):
        return ""
    s = str(s)
    # Remove Byte Order Mark (BOM) and zero-width spaces/invisible separators
    s = s.replace("\ufeff", "")
    s = re.sub(r"[\u200B-\u200D\uFEFF]", "", s)
    return s.strip()

def sanitize_filename(name: str) -> str:
    """Convert a string to a filesystem-safe format by replacing special characters."""
    name = clean_text(name)
    return re.sub(r"[^\w\-]", "_", name)

def safe_name(name: str) -> str:
    """Normalize city names for simple matching (lowercase, remove accents)."""
    if name is None or pd.isna(name):
        return ""
    name = clean_text(name)
    # Collapse multiple spaces to one
    name = re.sub(r'\s+', ' ', name)
    # Remove accents/diacritics and encode to ASCII
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    return name.lower().strip()

def canonicalize(s: str) -> str:
    """Strong normalization for robust fuzzy name matching (removes punctuation, spaces, accents)."""
    if s is None:
        return ""
    s = str(s).lower().strip()
    # Decompose characters and remove combining marks (accents)
    s = unicodedata.normalize('NFKD', s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # Replace common separators with spaces
    s = re.sub(r'[_\[\]\(\)/\\]', ' ', s)
    # Replace remaining non-word/non-space characters with space
    s = re.sub(r'[^\w\s]', ' ', s)
    # Collapse multiple spaces to one and strip
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def normalize_name(x) -> str | None:
    """Applies canonicalization and sanitization steps for generating a normalized name."""
    if pd.isna(x):
        return None
    # Sanitize, then clean, then remove leading '_' resulting from sanitization
    return clean_text(sanitize_filename(x)).lstrip("_")

# =============================================================================
# XARRAY AND CORDEX UTILITIES
# =============================================================================

def count_urban_cells(file: str, debug: bool = False) -> int:
    """
    Count the number of urban cells (where 'urmask' equals 1) in a NetCDF file.
    
    Args:
        file: Path to the NetCDF file containing the 'urmask' variable.
    
    Returns:
        The total count of urban cells.
    """
    ds = xr.open_dataset(file)
    # Retrieve the urban mask variable values
    urmask = ds['urmask'].values
    # Sum the number of cells where urmask is 1
    nurban = int(np.sum(urmask == 1))
    ds.close()
    return nurban

def create_cordex_grid(domain_id: str) -> xr.Dataset:
    """
    Generates an Xarray Dataset containing a CORDEX grid definition 
    (including 2D coordinates and boundaries).
    """
    # Load the CORDEX domain definition
    grid = cx.domain(domain_id, bounds=True, mip_era="CMIP6")
    
    # Use cf_xarray to calculate the corner vertices from the bounds arrays
    lon_b = cf_xarray.bounds_to_vertices(
        grid.vertices_lon, bounds_dim="vertices", order="counterclockwise"
    )
    lat_b = cf_xarray.bounds_to_vertices(
        grid.vertices_lat, bounds_dim="vertices", order="counterclockwise"
    )
    
    # Add the 2D boundary coordinates to the dataset
    return grid.assign_coords(lon_b=(("y", "x", "vertices"), lon_b.values), 
                              lat_b=(("y", "x", "vertices"), lat_b.values))

def get_lat_lon_names(ds: xr.Dataset) -> tuple[str, str]:
    """
    Detects the names of the latitude and longitude coordinates in the Dataset 
    based on common conventions.
    """
    lon_candidates = ["lon", "longitude", "nav_lon", "x", "rlon"]
    lat_candidates = ["lat", "latitude", "nav_lat", "y", "rlat"]
    
    # Find the first candidate that exists in the dataset's coordinates
    lon_name = next((c for c in lon_candidates if c in ds.coords), None)
    lat_name = next((c for c in lat_candidates if c in ds.coords), None)
    
    if lon_name is None or lat_name is None:
        raise KeyError(f"Could not find lon/lat coordinates in {list(ds.coords)}")
    
    return lat_name, lon_name

## Parameters

In [ ]:
urban_var = "sfturf"

urban_sur_th = 5
orog_diff = 100
sftlf_th = 70
ratio_r2u = 2
min_city_size = 20
lon_lim = 1
lat_lim = 1

variable = "tasmin"
period_star = "1991"
period_stop = "2020"
period = slice(period_star, period_stop)

mip_era = "CMIP6"
driving_source_id = "ERA5"
frequency = "mon"
variable= 'tasmin'
domain = "EUR-11"
model = "REMO"
domain_resolution = int(domain.split("-")[1])

In [ ]:
output_folder = "urban_th_sensitivity"
os.makedirs(output_folder, exist_ok=True)  # crea carpeta si no existe

### Load the data

In [ ]:
xarray_open_kwargs['chunks'] = {'time': 160}

dsets = open_datasets(
    [variable],
    frequency=frequency,
    driving_source_id=driving_source_id,
    mask=True,
    add_missing_bounds=False, 
    source_id =['RACMO23E', 'ICON-CLM-202407-1-1', 'CCLM6-0-1-URB', 'HCLIM43-ALADIN', 'ROAM-NBS', 'REMO2020-2-2-iMOVE-LUC', 'REMO2020-2-2-MR2', 'REMO2020-2-2-TEB', 'REMO2020-2-2', 'REMO2020-2-2-iMOVE', 'WRF451Q', 'CNRM-ALADIN64E1', 'RegCM5-0', 'ALARO1-SFX']
    
)

In [ ]:
# delete simulatios with not urban static variable
to_delete = [dset for dset, ds in dsets.items() if urban_var not in ds.variables]

for dset in to_delete:
    del dsets[dset]

In [ ]:
for dset in dsets.keys():
    dsets[dset] = dsets[dset].sel(time=period)

In [ ]:
for dset in dsets.keys():
    if not check_equal_period(dsets[dset], period):
        print(f"Temporal coverage of {dset} does not match with {period}")

In [ ]:
for dset in dsets.keys():
    dsets[dset] = standardize_unit(dsets[dset], variable)

In [ ]:
print(dsets.keys())

### Load the cities

In [ ]:
import geopandas as gpd

# Define paths for auxiliary data and UCDB geopackage
root_aux_data = "/mnt/CORDEX_CMIP6_tmp/aux_data/"
ucdb_path = f"{root_aux_data}GHS_FUA_UCD/GHS_UCDB_GLOBE_R2024A.gpkg"

# Load UCDB centroids and general characteristics layers
gdf_centroids = gpd.read_file(ucdb_path, layer="UC_centroids")
gdf_char = gpd.read_file(ucdb_path, layer="GHS_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A")

# Clean column names (remove leading/trailing spaces or hidden characters)
gdf_centroids.columns = gdf_centroids.columns.str.strip().str.replace("﻿", "", regex=True)
gdf_char.columns = gdf_char.columns.str.strip().str.replace("﻿", "", regex=True)

# Reproject centroids to WGS84 (EPSG:4326) for latitude/longitude coordinates
gdf_centroids = gdf_centroids.to_crs(epsg=4326)

# Merge centroids geometry with general characteristics using ID_UC_G0 as key
gdf_merged = gdf_char.merge(
    gdf_centroids[["ID_UC_G0", "geometry"]],
    on="ID_UC_G0"
)

# Set the merged geometry to the centroid geometry (geometry_y)
gdf_merged = gdf_merged.set_geometry("geometry_y")

# Extract longitude and latitude for filtering
lon_tmp = gdf_merged.geometry.x
lat_tmp = gdf_merged.geometry.y

# Filter cities by minimum area (12x12 km² = 144 km²)
gdf_filtered = gdf_merged[gdf_merged["GC_UCA_KM2_2025"] >= 12.5*12.5].copy()

# Filter by bounding box covering Europe
gdf_europe = gdf_filtered[
    (lat_tmp >= 34) & (lat_tmp <= 72) &
    (lon_tmp >= -25) & (lon_tmp <= 45)
].copy()

# Display the number of filtered cities and a preview of selected columns
print(f"Filtered cities in Europe >=144 km²: {len(gdf_europe)}")
print(gdf_europe[["ID_UC_G0", "GC_UCA_KM2_2025"]].head())

In [ ]:
thresholds = [30, 40, 50, 60]

In [ ]:
gdf_europe = gdf_europe[gdf_europe["GC_UCN_MAI_2025"].apply(clean_text) == "Manchester"].copy()
lon_lim= 0.75
lat_lim = 0.75
print(gdf_europe)

## Create the urmask for each city and threshold

In [ ]:
# =============================================================================
# PREPARE EURO-CORDEX MODELS
# =============================================================================
csv_path = os.path.join(output_folder, "skipped_cities.csv")

gdf_europe = gdf_europe.to_crs(epsg=4326)
skip_list = []

for row in gdf_europe.itertuples():
    city_name = str(row.GC_UCN_MAI_2025).strip()
    country_name = str(row.GC_CNT_GAD_2025).strip()

    if not city_name or not country_name:
        print(f"Skipping city with empty name or country: {row}")
        skip_list.append({"city": city_name, "country": country_name})
        continue

    lon, lat = row.geometry_y.x, row.geometry_y.y  

    safe_city = sanitize_filename(city_name)
    safe_country = sanitize_filename(country_name)
    print(f"Processing: {city_name}, {country_name} ({lat:.3f}, {lon:.3f})")

    # Centroid of the geoseries
    ucdb_city_gs = gpd.GeoSeries([row.geometry_x], crs="ESRI:54009").to_crs(epsg=4326)
    geom = ucdb_city_gs.iloc[0]    

    # ----------------- CREATE SUBPLOTS -----------------
    fig, axes = plt.subplots(
        nrows=len(dsets), ncols=len(thresholds),
        figsize=(4.5 * len(thresholds), 4.5 * len(dsets)),
        squeeze=False
    )
    im = None

    for m_idx, (model_name, ds) in enumerate(dsets.items()):
        for t_idx, urban_th in enumerate(thresholds):
            ax = axes[m_idx, t_idx]

            # Prepare the dataset
            ds = fix_360_longitudes(ds, lonname="lon")
            ds_sfturf = ds[[urban_var]].compute()
            ds_orog   = ds[["orog"]].compute()
            ds_sftlf  = ds[["sftlf"]].compute()

            try:
                rcm_name = ds.attrs.get("model_id", ds.attrs.get("source_id", "")) + "_" + \
                           ds.attrs.get("rcm_version_id", ds.attrs.get("version_realization", ""))
            except Exception:
                rcm_name = model_name

            URBAN = UrbanVicinity(
                urban_sur_th= urban_sur_th,
                orog_diff=orog_diff,
                sftlf_th=sftlf_th,
                ratio_r2u=ratio_r2u,
                min_city_size=min_city_size,
                lon_city=lon,
                lat_city=lat,
                lon_lim=lon_lim,
                lat_lim=lat_lim,
                model=model,
                domain=domain,
                urban_th=urban_th,
                urban_var=urban_var,
            )
            ds_sfturf = URBAN.crop_area_city(ds=ds_sfturf, res=domain_resolution)
            ds_orog = URBAN.crop_area_city(ds=ds_orog, res=domain_resolution)
            ds_sftlf = URBAN.crop_area_city(ds=ds_sftlf, res=domain_resolution)                

            if t_idx == 0:
                # Always show the model name, even if axis will be turned off
                ax.text(
                    -0.15, 0.5,                               # slightly outside left side
                    rcm_name if rcm_name else "Unknown model",
                    fontsize=18, rotation=90,
                    va='center', ha='center',
                    transform=ax.transAxes
                )
            

            try:
                sfturf_mask, sfturf_sur_mask, orog_mask, sftlf_mask = URBAN.define_masks(
                    ds_sfturf=ds_sfturf,
                    ds_orog=ds_orog,
                    ds_sftlf=ds_sftlf,
                )
            except ValueError as e:
                print(f"{rcm_name}: {e} skipping plot")
                ax.axis("off")
                continue

            urmask = URBAN.select_urban_vicinity(
                sfturf_mask=sfturf_mask,
                orog_mask=orog_mask,
                sftlf_mask=sftlf_mask,
                sfturf_sur_mask=sfturf_sur_mask,
            )

            urmask_values = urmask['urmask'].values
            if np.any(urmask_values == 1): 
                lat_urban = urmask.lat.values[urmask_values == 1] 
                lon_urban = urmask.lon.values[urmask_values == 1] 
                min_dist = min(haversine(lat, lon, la, lo) 
                               for la, lo in zip(lat_urban, lon_urban)) 
            print(f"{rcm_name}: min_dist = {min_dist:.2f} km") 
            
            if min_dist >= 25: 
                print(f"Skipping subplot for {rcm_name} — min_dist {min_dist:.1f} km") 
                ax.axis("off") 
                continue

            ucdb_city_gs.plot(ax=ax, facecolor="none", edgecolor="#ff66ff", linewidth=4, zorder=100)

            im = ax.pcolormesh(
                ds_sfturf.lon, ds_sfturf.lat, ds_sfturf[urban_var],
                cmap='binary', vmin=0, vmax=100
            )
            plot_urban_polygon(urmask, ax)
            ax.set_xticks([])
            ax.set_yticks([])


    # ------------------- TITLES & LABELS -------------------
    fig.suptitle(f"{clean_text(city_name)}, {clean_text(country_name)}", fontsize=28, fontweight="bold", y=0.95)

    # Threshold labels slightly below the top, centered over each column
    for t_idx, urban_th in enumerate(thresholds):
        fig.text(
            0.10 + (t_idx + 0.5) * (0.82 / len(thresholds)),
            0.93,
            f"th = {clean_text(urban_th)}",
            fontsize=22, va='center', ha='center'
        )

    # ------------------- COLORBAR -------------------
    if im is not None:
        # Colorbar vertical at right
        cbar_ax = fig.add_axes([0.92, 0.4, 0.02, 0.35])
        cbar = fig.colorbar(im, cax=cbar_ax, orientation="vertical")
        cbar.ax.set_title(clean_text(urban_var), fontsize=28, pad=8, loc='center')
        cbar.set_ticks([0, 25, 50, 75, 100])
        cbar.ax.tick_params(labelsize=24)

    # Layout
    fig.subplots_adjust(left=0.12, right=0.9, top=0.93, bottom=0.1, wspace=0.25, hspace=0.25)

    # ------------------- SAVE FIGURES -------------------
    outdir_plots = os.path.join(output_folder, "results", "plots", "urmask")
    os.makedirs(outdir_plots, exist_ok=True)

    filepath_pdf = os.path.join(outdir_plots, f"{safe_city}_{safe_country}.pdf")
    fig.savefig(filepath_pdf, format="pdf",  bbox_inches='tight')
    print(f"Saved: {filepath_pdf}")

    filepath_png = os.path.join(outdir_plots, f"{safe_city}_{safe_country}.png")
    fig.savefig(filepath_png, format="png", dpi=300,  bbox_inches='tight')
    print(f"Saved: {filepath_png}")

    fig.show()

if skip_list:
    pd.DataFrame(skip_list).to_csv(csv_path, index=False)
    print(f"Skipped {len(skip_list)} cities — saved to {csv_path}")


## Plot maps for threshold

In [ ]:
# Configuration
urmask_root = os.path.join(output_folder, "results", 'datasets', "urmask")
thresholds = ["Thr30", "Thr40", "Thr50", "Thr60", "ucdb"]   # Urban Fractions
agreement_levels = [5, 6, 7, 8,9]                    # exact number of models
gdf_cities = gdf_europe.copy()

# Get model list from folders
models = [
    d for d in os.listdir(os.path.join(urmask_root, thresholds[0]))
    if os.path.isdir(os.path.join(urmask_root, thresholds[0], d))
]
n_models = len(models)
print(f"Found {n_models} models: {models}")

# Collect urmask presence data 
presence = {}  # dict {threshold: {city: count}}
for thr in thresholds:
    presence[thr] = {}
    for model in models:
        path = os.path.join(urmask_root, thr, model, "*.nc")
        files = glob.glob(path)
        for f in files:
            city_id = os.path.basename(f).replace(".nc", "")
            presence[thr].setdefault(city_id, 0)
            presence[thr][city_id] += 1

# Prepare dataframe to save
records = []

# Plot matrix
fig, axes = plt.subplots(
    nrows=len(thresholds), ncols=len(agreement_levels),
    figsize=(10*len(agreement_levels), 10*len(thresholds)),
    subplot_kw={"projection": ccrs.PlateCarree()},
    squeeze=False,
)

for r_idx, thr in enumerate(thresholds):
    for c_idx, agree in enumerate(agreement_levels):
        ax = axes[r_idx, c_idx]
        ax.set_title(f"{thr} — at least {agree}/{n_models} models", fontsize=14)
        ax.coastlines(resolution="50m", linewidth=0.8)

        # select cities that match EXACTLY the criterion
        valid_cities = [
            city for city, count in presence[thr].items() if count >= agree
        ]

        if valid_cities:
            gdf_plot = gdf_cities[gdf_cities.apply(
                lambda row: f"{sanitize_filename(row['GC_UCN_MAI_2025'])}_{sanitize_filename(row['GC_CNT_GAD_2025'])}" in valid_cities,
                axis=1
            )]
            # ensure points
            if not all(gdf_plot.geometry.type == "Point"):
                gdf_points = gdf_plot.copy()
                gdf_points["geometry"] = gdf_points.geometry.centroid
            else:
                gdf_points = gdf_plot

            # plot points manually on Cartopy axes
            xs = gdf_points.geometry.x
            ys = gdf_points.geometry.y
            ax.scatter(xs, ys, color="red", s=45, transform=ccrs.PlateCarree())

            # labels and save to dataframe 
            for x, y, label, country in zip(
                    gdf_points.geometry.x,
                    gdf_points.geometry.y,
                    gdf_points["GC_UCN_MAI_2025"],
                    gdf_points["GC_CNT_GAD_2025"]
            ):
                safe_label = clean_text(label)
                safe_country = clean_text(country)
                ax.text(x, y, safe_label, fontsize=8, ha="left", transform=ccrs.PlateCarree())

            
                records.append({
                    "threshold": thr,
                    "models": agree,
                    "city": safe_label,
                    "country": safe_country,
                    "lon": x,
                    "lat": y
                })
            
            ax.set_title(clean_text(f"{thr} — at least {agree}/{n_models} models"), fontsize=14)


        ax.set_extent([-25, 45, 34, 72])
        ax.set_axis_off()

plt.subplots_adjust(wspace=0.02, hspace=0.02)
plt.tight_layout(pad=0.2)

# Save figure
output_pdf = os.path.join(output_folder, "results/plots/matrix_cities_thr_vs_models.pdf")
plt.savefig(output_pdf, format="pdf", bbox_inches="tight")
print(f"Figure saved at: {output_pdf}")

# Save CSV 
df_cities = pd.DataFrame(records)
output_csv = os.path.join(output_folder, "results/matrix_cities_thr_vs_models.csv")
df_cities.to_csv(output_csv, index=False)
print(f"CSV saved at: {output_csv}")


### Creata a dataframe with the  summary data for the plots

In [ ]:
# Configuration 
urmask_root = os.path.join(output_folder, "results", 'datasets', "urmask")
thresholds = ["Thr30", "Thr40", "Thr50", "Thr60", "ucdb"]
models = [d for d in os.listdir(os.path.join(urmask_root, thresholds[0])) if os.path.isdir(os.path.join(urmask_root, thresholds[0], d))]
n_models = len(models)

# Collect urmask presence
presence = {}  # dict {threshold: {city_id: count}}
for thr in thresholds:
    presence[thr] = {}
    for model in models:
        path = os.path.join(urmask_root, thr, model, "*.nc")
        files = glob.glob(path)
        for f in files:
            city_id = os.path.basename(f).replace(".nc", "")
            presence[thr].setdefault(city_id, 0)
            presence[thr][city_id] += 1

# Create summary DataFrame including lat/lon
rows = []
for thr in thresholds:
    for city_id, count in presence[thr].items():
        # Split city_id correctly
        if city_id.startswith("_"):
            city_id_clean = city_id[1:] 
        else:
            city_id_clean = city_id
        if "__" in city_id_clean:
            safe_city, safe_country = city_id_clean.split("__", 1)
        elif "_" in city_id_clean:
            safe_city, safe_country = city_id_clean.rsplit("_", 1)
        else:
            safe_city, safe_country = city_id_clean, ""

        # Get original lat/lon from new UCDB columns
        geom_row = gdf_europe[
            (gdf_europe['GC_UCN_MAI_2025'].apply(normalize_name) == clean_text(safe_city)) &
            (gdf_europe['GC_CNT_GAD_2025'].apply(normalize_name) == clean_text(safe_country))
        ]


        if not geom_row.empty:
            point = geom_row.geometry_y.values[0]  
            lon, lat = point.x, point.y            
            city_name = clean_text(geom_row["GC_UCN_MAI_2025"].values[0])
            country_name = clean_text(geom_row["GC_CNT_GAD_2025"].values[0])

        else:
            lat = lon = city_name = country_name = None
        
        rows.append({
            "threshold": thr,
            "city": city_name if city_name else "UNKNOWN",
            "country": country_name if country_name else "UNKNOWN",
            "city_id": city_id,
            "n_models_with_urmask": count,
            "lat": lat,
            "lon": lon
        })


df_summary = pd.DataFrame(rows)

# Save CSV
csv_path = os.path.join(output_folder, "results", "cities_by_threshold_models.csv")
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df_summary.to_csv(csv_path, index=False)
print(f"CSV saved at: {csv_path}")
print(df_summary)

## Plot maps for each threshold with number of cells

In [ ]:
# Configuration
thresholds = ["Thr30", "Thr40", "Thr50", "Thr60", 'ucdb']
csv_path = os.path.join(output_folder, "results", "cities_by_threshold_models.csv")
df_all = pd.read_csv(csv_path)

# Cmap with 9 different colors
colors = ["#ffffcc", "#ffeda0", "#fed976", "#feb24c",
          "#fd8d3c", "#fc4e2a", "#e31a1c", "#b10026", '#000000']
cmap = ListedColormap(colors)
bounds = [0.5 + i for i in range(10)]  # 0.5, 1.5, ... 8.5
norm = BoundaryNorm(bounds, cmap.N)


os.makedirs(os.path.join(output_folder, "results", "maps_by_threshold"), exist_ok=True)

for thr in thresholds:
    df_th = df_all[df_all["threshold"] == thr]

    fig = plt.figure(figsize=(12, 12))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_title(f"Cities for {thr}", fontsize=20)

    # Coastlines and borders
    ax.coastlines(resolution="50m", linewidth=1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="gray")

    # Plot cities
    for _, row in df_th.iterrows():
        lat = row["lat"]
        lon = row["lon"]
        if pd.notnull(lat) and pd.notnull(lon):
            n_models = row['n_models_with_urmask']
            color = cmap(norm(n_models))
            ax.plot(lon, lat, 'o', color=color, markersize=8, transform=ccrs.PlateCarree(), zorder=5)

    ax.set_extent([-25, 45, 34, 72])
    ax.set_axis_off()

    # Colorbar to the right
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # [left, bottom, width, height]
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')
    cbar.set_label("Number of models with urmask", fontsize=12)
    cbar.set_ticks(range(1, 10))

    # Save PDF
    output_pdf = os.path.join(output_folder, "results", "maps_by_threshold", f"{thr}_cities.pdf")
    plt.savefig(output_pdf, format="pdf", bbox_inches="tight")
    print(f"Map for {thr} saved at: {output_pdf}")

In [ ]:
gdf_europe['city_id'] = gdf_europe.apply(
    lambda row: f"{clean_text(sanitize_filename(row['GC_UCN_MAI_2025']))}_{clean_text(sanitize_filename(row['GC_CNT_GAD_2025']))}",
    axis=1
)

In [ ]:
# Configuration
thresholds = ["Thr30", "Thr40", "Thr50", "Thr60", "ucdb"]  # Thresholds to iterate over
csv_path = os.path.join(output_folder, "results", "cities_by_threshold_models.csv")
df_all = pd.read_csv(csv_path)  # Table of cities by threshold and models

# Define custom color map and normalization for number of models
colors = ["#ffffcc", "#ffeda0", "#fed976", "#feb24c",
          "#fd8d3c", "#fc4e2a", "#e31a1c", "#b10026", "#000000"]
cmap = ListedColormap(colors)
bounds = [0.5 + i for i in range(10)]  # Boundaries between color intervals
norm = BoundaryNorm(bounds, cmap.N)

# Make sure gdf_europe is in WGS84 (EPSG:4326)
gdf_europe = gdf_europe.to_crs(epsg=4326)

# Create output directory for generated maps
os.makedirs(os.path.join(output_folder, "results", "maps_by_threshold_polygons"), exist_ok=True)

# Loop over each threshold and plot the polygons
for thr in thresholds:
    # Filter DataFrame for the current threshold
    df_th = df_all[df_all["threshold"] == thr]

    # Set up map figure with PlateCarree projection
    fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_title(f"Cities polygons for {thr}", fontsize=20)

    # Draw coastlines and national borders
    ax.coastlines(resolution="50m", linewidth=1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="gray")

    # Iterate over each city in the filtered DataFrame
    for _, row in df_th.iterrows():
        city_id = row['city_id']
        geom_row = gdf_europe[gdf_europe['city_id'] == city_id]

        if not geom_row.empty:
            # Select color based on number of models with urmask
            color = cmap(norm(row['n_models_with_urmask']))

            # Get the polygon geometry (geometry_x column)
            geom = geom_row.iloc[0].geometry_x

            # Convert from original CRS to WGS84 for plotting
            geom_gs = gpd.GeoSeries([geom], crs="ESRI:54009")  # Original CRS of your data
            geom_gs = geom_gs.to_crs(epsg=4326)                # Convert to WGS84
            geom = geom_gs.iloc[0]

            # Add polygon to the map
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor=color, edgecolor=color, alpha=0.8)

    # Set map extent to Europe and remove axis ticks
    ax.set_extent([-25, 45, 34, 72])
    ax.set_axis_off()

    # Add colorbar showing number of models with urmask
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')
    cbar.set_label("Number of models with urmask", fontsize=12)
    cbar.set_ticks(range(1, 10))

    # Save figure as PDF
    output_pdf = os.path.join(output_folder, "results", "maps_by_threshold_polygons", f"{thr}_cities_polygons.pdf")
    plt.savefig(output_pdf, format="pdf", bbox_inches="tight")
    print(f"Polygon map for {thr} saved at: {output_pdf}")

## Bar plot summary

In [ ]:
# Load the data
csv_path = os.path.join(output_folder, "results", "cities_by_threshold_models.csv")
df_all = pd.read_csv(csv_path)

# Output folder for the figure
os.makedirs(os.path.join(output_folder, "results", "summary_plots"), exist_ok=True)

# Thresholds and colors (light to dark)
thresholds = ["Thr30", "Thr40", "Thr50", "Thr60"]
colors = ["#ffeda0", "#feb24c", "#fd8d3c", "#e31a1c"]  # Light to dark per threshold

# Cap maximum number of models at 9 to avoid extreme values
df_all["n_models_with_urmask"] = df_all["n_models_with_urmask"].clip(upper=9)

# Pivot DataFrame to have cities as rows and thresholds as columns
pivot_df = df_all.pivot_table(
    index="city_id",
    columns="threshold",
    values="n_models_with_urmask",
    fill_value=0
)

# Clean up city_id by removing leading underscores
pivot_df.index = pivot_df.index.str.lstrip("_")

# Sort cities by total models across all thresholds
pivot_df["total"] = pivot_df.sum(axis=1)
pivot_df.sort_values("total", ascending=False, inplace=True)
pivot_df.drop(columns="total", inplace=True)

# Create the figure
fig, ax = plt.subplots(figsize=(50, 10))  # Wide figure for many cities

# Plot overlapping bars per threshold
for z, (thr, color) in enumerate(zip(thresholds, colors)):
    ax.bar(
        pivot_df.index,
        pivot_df[thr],
        color=color,
        edgecolor="black",
        label=thr,
        zorder=z  # Lower zorder = drawn first (behind)
    )

# Style and labels
ax.set_xlabel("City", fontsize=14)
ax.set_ylabel("Number of models with urmask", fontsize=14)
ax.set_title("Number of models with urmask per city by threshold", fontsize=16)

# Rotate x-axis labels for better readability
ax.tick_params(axis='x', rotation=80, labelsize=8)

# Adjust x tick labels to align properly
ax.set_xticks(range(len(pivot_df.index)))
ax.set_xticklabels(pivot_df.index, ha='right')

# Legend
ax.legend(title="Threshold")

# Tidy layout
plt.tight_layout()

# Save plot as PDF
output_pdf = os.path.join(output_folder, "results", "summary_plots", "stacked_bars_cities_thresholds_overlap.pdf")
plt.savefig(output_pdf, format="pdf", bbox_inches="tight")
print(f"Overlapping bar plot saved as PDF at: {output_pdf}")


In [ ]:
rotated_grid = create_cordex_grid("EUR-11")  


## Sum of urban fraction for Europe, Lodon and Paris

In [ ]:
target_grid = xr.Dataset(
    {
        "lat": (["y", "x"], rotated_grid.lat_b.values),
        "lon": (["y", "x"], rotated_grid.lon_b.values),
    }
)

total_urmask = None # Initialize the accumulator for the urban fraction sum

# =============================================================================
# ACCUMULATE REGRIDDED URBAN MASK (urban_var) ACROSS ALL MODELS
# =============================================================================
for model_name, ds in dsets.items():
    # 1. Standardize Longitudes: Convert 0-360 longitude coordinates to -180-180 
    # to ensure consistency before regridding.
    ds = fix_360_longitudes(ds, lonname="lon")
    
    # Extract the urban variable (e.g., land surface type mask) and compute it
    ds_sfturf = ds[[urban_var]].compute()

    # 2. Create Regridder: Initialize the xESMF Regridder object.
    # 'nearest_s2d' (Nearest-neighbor source to destination) is used to map 
    # the urban fraction from the source model grid to the target grid.
    regridder = xe.Regridder(ds_sfturf, target_grid, "nearest_s2d", reuse_weights=False)
    
    # 3. Regrid/Interpolate: Apply the regridding process.
    ds_interp = regridder(ds_sfturf)
    
    # 4. Accumulate: Sum the regridded urban fraction to the total mask
    if total_urmask is None:
        total_urmask = ds_interp.copy()
    else:
        total_urmask += ds_interp
        
# Extract the accumulated urban variable (e.g., "urmask") from the total dataset
total_urmask_var = total_urmask[urban_var]

### Plot for Europe

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

# Plot the accumulated urban fraction using Xarray's built-in plotting functionality
im = total_urmask_var.plot(
    cmap="Greys", # Use a grayscale colormap to visualize accumulated fraction
    cbar_kwargs={"label": "Accumulated urban fraction", "shrink": 0.8},
    ax=ax
)

# Enhance Axes Labels
ax.set_xlabel("Longitude", fontsize=14)
ax.set_ylabel("Latitude", fontsize=14)

# Remove the default automatic title (often redundant)
ax.set_title("")

# =============================================================================
# SAVE AND DISPLAY OUTPUT
# =============================================================================
# Define output directory path (assuming 'Path' object is imported)
output_dir = Path("urban_th_sensitivity/results/plots")

# Ensure the output directory exists
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "accumulated_urban_fraction.pdf"

# Save the figure as a PDF file
plt.savefig(output_path, format="pdf", bbox_inches="tight")

# Display the plot
plt.show()

# Print confirmation message
print(f"Plot saved to: {output_path}")

In [ ]:
# --- 1. Definir centroide de Londres y buffer ---
# Coordenadas aproximadas de Londres
lon_paris, lat_paris = 2.35,  48.85


# Definimos bounding box (ej: 3 grados alrededor)
buffer_deg = 1
lat_min, lat_max = lat_paris - buffer_deg, lat_paris + buffer_deg
lon_min, lon_max = lon_paris - buffer_deg, lon_paris + buffer_deg

# --- 2. Crear target grid recortado ---
# rotated_grid ya tiene lat_b/lon_b en 2D
target_grid_full = xr.Dataset(
    {
        "lat": (["y", "x"], rotated_grid.lat_b.values),
        "lon": (["y", "x"], rotated_grid.lon_b.values),
    }
)

# recorte espacial
mask = (
    (target_grid_full.lat >= lat_min) & (target_grid_full.lat <= lat_max) &
    (target_grid_full.lon >= lon_min) & (target_grid_full.lon <= lon_max)
)

target_grid = target_grid_full.where(mask, drop=True)

# --- 3. Sumar urmask interpolado en zona de Londres ---
total_urmask = None

for model_name, ds in dsets.items():
    # Normalizar longitudes a [-180,180]
    ds = fix_360_longitudes(ds, lonname="lon")
    
    # Seleccionamos solo la variable urbana
    ds_sfturf = ds[[urban_var]].compute()

    # --- dentro del loop ---
    lat_name, lon_name = get_lat_lon_names(ds_sfturf)
    # Recorte espacial usando los nombres reales
    ds_sfturf_crop = ds_sfturf.where(
        (ds_sfturf['lon'] >= lon_min) & (ds_sfturf['lon'] <= lon_max) &
        (ds_sfturf['lat'] >= lat_min) & (ds_sfturf['lat'] <= lat_max),
        drop=True
    )


    # Creamos el regridder
    regridder = xe.Regridder(ds_sfturf, target_grid, "nearest_s2d", reuse_weights=False)

    # Regrid/interp
    ds_interp = regridder(ds_sfturf)

    # Sumar acumulativamente
    if total_urmask is None:
        total_urmask = ds_interp.copy()
    else:
        total_urmask += ds_interp


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# Nos quedamos con la variable (ej. "urmask")
total_urmask_var = total_urmask[urban_var]

# Crear figura grande
fig, ax = plt.subplots(figsize=(12, 9))

# Plot
im = total_urmask_var.plot(
    cmap="Greys", 
    cbar_kwargs={"label": "Accumulated urban fraction", "shrink": 0.8},
    ax=ax
)

# Mejorar ejes
ax.set_xlabel("Longitude", fontsize=14)
ax.set_ylabel("Latitude", fontsize=14)

# Quitar título automático
ax.set_title("")

# Ruta de salida
output_dir = Path("urban_th_sensitivity/results/plots")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir /"paris_accumulated_urban_fraction.pdf"

# Guardar en PDF
plt.savefig(output_path, format="pdf", bbox_inches="tight")

plt.show()
print(f"Plot guardado en: {output_path}")


## Plot number of grid cells classified as urban for different urban fraction thresholds

In [ ]:
# Configuration and paths
output_folder = "urban_th_sensitivity/results"
csv_path = os.path.join(output_folder, "cities_by_threshold_models.csv")
base_path = os.path.join(output_folder, "datasets/urmask")
plot_folder = os.path.join(output_folder, "plots")
os.makedirs(plot_folder, exist_ok=True)

thresholds = ["Thr30", "Thr40", "Thr50", "Thr60"]



# Read cities table
cities_df = pd.read_csv(csv_path)
cities_df['city_country'] = cities_df['city'] + ", " + cities_df['country']
cities_df['city_country_norm'] = cities_df['city_country'].apply(safe_name)

records = []

# Collect urban cells per threshold and model
for thr in thresholds:
    thr_folder = os.path.join(base_path, thr)
    models = [m for m in os.listdir(thr_folder) if os.path.isdir(os.path.join(thr_folder, m))]
    for model in models:
        files = glob.glob(os.path.join(thr_folder, model, "*.nc"))
        for f in files:
            fname = os.path.basename(f).replace(".nc", "")
            city_row = cities_df[cities_df['city_id'] == fname]
            if city_row.empty:
                continue
            city_country = city_row.iloc[0]['city_country']
            city_country_norm = city_row.iloc[0]['city_country_norm']
            nurban = count_urban_cells(f)  # your function
            records.append({
                "model": model,
                "threshold": thr,
                "city_country": city_country,
                "city_country_norm": city_country_norm,
                "urban_cells": nurban
            })

# UCDB from gdf_europe
area_cell = 12.5 * 12.5  # km² per cell
for row in gdf_europe.itertuples():
    city_name = str(row.GC_UCN_MAI_2025).strip()
    country_name = str(row.GC_CNT_GAD_2025).strip()
    city_country = f"{city_name}, {country_name}"
    city_country_norm = safe_name(city_country)
    city_area = row.GC_UCA_KM2_2025
    urban_cells = city_area / area_cell
    if urban_cells >= 1:
        records.append({
            "model": "UCDB",
            "threshold": "ucdb",
            "city_country": city_country,
            "city_country_norm": city_country_norm,
            "urban_cells": round(urban_cells, 1)  # keep one decimal
        })

# Create DataFrame from records
df = pd.DataFrame(records)

# UCDB series indexed by normalized names
ucdb_df = df[df['threshold'] == 'ucdb'][['city_country_norm','urban_cells','city_country']]

# Only keep cities with UCDB >= 1
valid_cities = ucdb_df[ucdb_df['urban_cells'] >= 1]['city_country_norm'].unique()

# Mapping from normalized name to pretty name
city_labels = ucdb_df.drop_duplicates('city_country_norm').set_index('city_country_norm')['city_country']

# Filter df to only valid cities
df = df[df['city_country_norm'].isin(valid_cities)]

# UCDB series for lookup
ucdb_series = ucdb_df.set_index('city_country_norm')['urban_cells']

# Plot per model
models_to_plot = df[df['threshold'] != 'ucdb']['model'].unique()

for model in models_to_plot:
    pivot_thr = df[df['model'] == model].pivot_table(
        index='city_country_norm',
        columns='threshold',
        values='urban_cells',
        fill_value=0
    )

    # Ensure pivot has exactly the valid cities in UCDB order
    pivot_thr = pivot_thr.reindex(valid_cities)

    # Add UCDB column
    pivot_thr['ucdb'] = pd.Series(
        [ucdb_series.get(x, 0) for x in pivot_thr.index],
        index=pivot_thr.index
    ).fillna(0)

    # Sort by UCDB descending
    pivot_thr = pivot_thr.sort_values('ucdb', ascending=False)

    x = np.arange(len(pivot_thr))
    fig, ax = plt.subplots(figsize=(25, 8))
    bar_width = 0.18
    colors = ['#fee08b', '#d9ef8b', '#fc8d59', '#d73027']
    thresholds_ordered = [c for c in ['Thr30', 'Thr40', 'Thr50', 'Thr60'] if c in pivot_thr.columns]

    # Bars for thresholds
    for i, col in enumerate(thresholds_ordered):
        ax.bar(
            x + bar_width,
            pivot_thr[col],
            color=colors[i],
            width=bar_width,
            label=col,
            edgecolor='lightgray'
        )

    # UCDB bar
    ax.bar(
        x + 2 * bar_width,
        pivot_thr['ucdb'],
        color='black',
        width=bar_width,
        label='UCDB',
        edgecolor='lightgray'
    )

    # X-axis labels
    ax.set_xticks(x + bar_width * len(thresholds_ordered) / 2)
    ax.set_xticklabels([clean_text(city_labels.get(c, "")) for c in pivot_thr.index], rotation=45, ha='right', fontsize=10)

    ax.set_ylabel("Urban cells count")
    ax.set_title(f"Urban cells for model {model}", fontsize=16)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.legend(title='Threshold')

    plt.tight_layout()
    outname = os.path.join(plot_folder, f'urban_cells_model_{sanitize_filename(model)}_ucdb.pdf')
    plt.savefig(outname, bbox_inches='tight')
    print(f"Saved plot: {outname}")


In [ ]:
bar_width_thr = 0.5

In [ ]:
ordered_cities = sorted(global_order, key=lambda x: ucdb_series.get(x, 0), reverse=True)
split_data = [x.split(", ") for x in ordered_cities]
df_table = pd.DataFrame(split_data, columns=["City", "Country"])

In [ ]:
# Plot Models + Summary
models_to_plot = df[df['threshold'] != 'ucdb']['model'].unique()
n_models = len(models_to_plot)

fig, axes = plt.subplots(
    n_models + 1, 1,
    figsize=(28, 3.5 * (n_models + 1)),
    sharex=True,
    gridspec_kw={'hspace': 0.08, 'left': 0.06, 'right': 0.96}
)
axes = np.atleast_1d(axes)

bar_width_thr = 0.5
bar_width_ucdb = 0.15
colors = ['#fee08b', '#d9ef8b', '#fc8d59', '#d73027']
thresholds_ordered = [t for t in ['Thr30', 'Thr40', 'Thr50', 'Thr60'] if t in thresholds]

x = np.arange(len(global_order))

# Define custom y-ticks for log scale
custom_yticks = [1, 2, 4, 10, 25]

# Individual Model Plots
for ax, model in zip(axes[:-1], models_to_plot):
    pivot_thr = df[df['model'] == model].pivot_table(
        index='city_country_norm',
        columns='threshold',
        values='urban_cells',
        fill_value=0
    ).reindex(valid_cities)

    pivot_thr['ucdb'] = ucdb_series.reindex(pivot_thr.index).fillna(0)
    pivot_thr = pivot_thr.sort_values('ucdb', ascending=False)

    for i, col in enumerate(thresholds_ordered):
        ax.bar(
            x, pivot_thr[col],
            color=colors[i],
            width=bar_width_thr,
            label=col if model == models_to_plot[0] else "",
            edgecolor='lightgray'
        )

    ax.bar(
        x + bar_width_ucdb * 1.5,
        pivot_thr['ucdb'],
        color='black',
        width=bar_width_ucdb,
        label='UCDB' if model == models_to_plot[0] else "",
        edgecolor='lightgray'
    )

    # Model name inside box (top-right)
    ax.text(
        0.985, 0.93, model,
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=20, 
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=3)
    )

    ax.set_ylabel("Nº. urban cells", fontsize=18, labelpad=10) 
    ax.yaxis.set_label_coords(-0.015, 0.5)
    ax.tick_params(axis='y', labelsize=16)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_yscale('log')

    # Apply custom y-ticks and labels
    ax.set_yticks(custom_yticks)
    ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_ylim(0, 40)

    # Remove space at plot edges
    ax.set_xlim(-0.5, len(x) - 0.5)

# Summary Plot
summary_ax = axes[-1]

df_all = pd.read_csv(csv_path)
df_all["n_models_with_urmask"] = df_all["n_models_with_urmask"].clip(upper=9)

pivot_df = df_all.pivot_table(
    index="city_id",
    columns="threshold",
    values="n_models_with_urmask",
    fill_value=0
)

# Normalize and align indices
pivot_df.index = pivot_df.index.map(safe_name)

pivot_df['ucdb_temp'] = ucdb_series.reindex(pivot_df.index).fillna(0)
pivot_df = pivot_df.sort_values('ucdb_temp', ascending=False).drop(columns=['ucdb_temp'])

# Fuzzy match pivot_df -> global_order
global_can = {g: canonicalize(g) for g in global_order}
pivot_can = {p: canonicalize(p) for p in pivot_df.index}

map_pivot_to_global = {}
threshold_sim = 0.85
for p_orig, p_can in pivot_can.items():
    best_g = None
    best_score = 0
    for g, g_can in global_can.items():
        score = SequenceMatcher(None, p_can, g_can).ratio()
        if score > best_score:
            best_score = score
            best_g = g
    if best_score >= threshold_sim:
        map_pivot_to_global[p_orig] = best_g

pivot_df = pivot_df.rename(index=map_pivot_to_global)
global_order = [g for g in global_order if g in pivot_df.index]
pivot_df = pivot_df.loc[sorted(global_order, key=lambda x: ucdb_series.get(x, 0), reverse=True)]

x_summary = np.arange(len(pivot_df))
summary_colors = colors

for thr, color in zip(["Thr30", "Thr40", "Thr50", "Thr60"], summary_colors):
    if thr not in pivot_df.columns:
        pivot_df[thr] = 0
    summary_ax.bar(
        x_summary,
        pivot_df[thr].values,
        width=bar_width_thr,
        edgecolor='lightgray',
        color=color,
        label=thr,
        zorder=2,
    )

summary_ax.set_ylabel("Nº. models", fontsize=18, labelpad=10)
summary_ax.yaxis.set_label_coords(-0.015, 0.5)
summary_ax.grid(axis='y', linestyle='--', alpha=0.4)
summary_ax.tick_params(axis='y', labelsize=16)
summary_ax.set_xticks(x_summary)
summary_ax.set_xticklabels(
    [clean_text(city_labels.get(c, "")) for c in pivot_df.index],
    rotation=45, ha='right', fontsize=16
)

# Remove space at plot edges for summary
summary_ax.set_xlim(-0.5, len(x_summary) - 0.5)

# Legend and Layout
handles, labels = axes[0].get_legend_handles_labels()

# Clean legend labels: remove "Thr" prefix
clean_labels = [label.replace("Thr", "") for label in labels]

fig.legend(
    handles, clean_labels, title='Thresholds',
    loc='upper center',
    ncol=len(thresholds_ordered) + 1,
    frameon=True, framealpha=1, fancybox=True, edgecolor='black',
    fontsize=20,          # larger font size
    title_fontsize=22,    # larger, bolder title
    bbox_to_anchor=(0.5, 0.92)  # adjust vertical placement if needed
)

plt.savefig(os.path.join(plot_folder, "urban_cells_all_models_ucdb_plus_summary.pdf"), bbox_inches='tight')
plt.savefig(os.path.join(plot_folder, "urban_cells_all_models_ucdb_plus_summary.png"), bbox_inches='tight')
print("\nSaved combined figure (models + summary).")